In [0]:
# Notebook autonome : il porte ses propres %run et se lance seul.
# Relancés par main_translations, ils sont sans effet de bord (idempotents).

In [0]:
%run ./translation_function

In [0]:
%run ./dim_trad_language

# build_translations



In [0]:
goods_species_translations = spark.table(f"{source_catalog}.goods_species_translations")
goods_varieties_translations = spark.table(f"{source_catalog}.goods_varieties_translations")
parameters_production_type_translations = spark.table(f"{source_catalog}.parameters_production_type_translations")
parameters_variables_translations = spark.table(f"{source_catalog}.parameters_variables_translations")
parameters_production_line_variables_translations = spark.table(f"{source_catalog}.parameters_production_line_variables_translations")
parameters_localizations_translations = spark.table(f"{source_catalog}.parameters_localizations_translations")
parameters_localization_groups_translations = spark.table(f"{source_catalog}.parameters_localization_groups_translations")
parameters_batch_note_categories_translations = spark.table(f"{source_catalog}.parameters_batch_note_categories_translations")
processes_phases_translations = spark.table(f"{source_catalog}.processes_phases_translations")

# Traductions du suivi des temps (DM-6516). Elles ne viennent pas de PostgreSQL
# mais du catalogue commun, et au format large : une colonne par langue.
trs_catalog = f"mal_maite_common_{current_environment}.gold"
trs_translation = spark.table(f"{trs_catalog}.trs_translation")


In [0]:
def cles_nomenclature(table, key_col, alias, label_col=None):
    """Clés actives d'une nomenclature, renommées comme dans la table de traduction.

    label_col : la colonne de code métier, qui servira de repli de dernier recours.
    Les nomenclatures conservent les lignes supprimées (drapeau deleted) : on les
    écarte ici, une entité supprimée n'ayant pas à être traduite.
    """
    cols = [F.col(key_col).alias(alias)]
    if label_col is not None:
        cols.append(F.col(label_col).cast("string").alias(FALLBACK_LABEL_COL))
    else:
        # Pas de code métier dans cette nomenclature : on retombe sur l'identifiant.
        cols.append(F.col(key_col).cast("string").alias(FALLBACK_LABEL_COL))

    return (
        spark.table(f"{current_catalog}.{common_schema}.{table}")
        .filter(F.col("deleted") == False)
        .select(*cols)
    )

In [0]:
# goods_species_translations -> dim_trad_specy
dim_trad_specy = build_translation_dim(
    goods_species_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_specy", "id_good_specy", "good_specy", "code"),
    key_cols=["good_specy"]
).withColumnRenamed("good_specy", "id_good_specy")

publish_dim(dim_trad_specy, "dim_trad_specy", ["id_good_specy", "language"], current_catalog + "." + common_schema)

In [0]:
# goods_varieties_translations -> dim_trad_variety
dim_trad_variety = build_translation_dim(
    goods_varieties_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_variety", "id_good_variety", "good_variety", "code"),
    key_cols=["good_variety"]
).withColumnRenamed("good_variety", "id_good_variety")

publish_dim(dim_trad_variety, "dim_trad_variety", ["id_good_variety", "language"], current_catalog + "." + common_schema)

In [0]:
# parameters_production_type_translations -> dim_trad_production_type
# La source nomme sa clé "id_parameters_production_type" (pluriel) alors que la
# table métier parameters_production_types porte "id_parameter_production_type"
# (singulier). On normalise ici sur le nom singulier, celui du modèle.
dim_trad_production_type = build_translation_dim(
    parameters_production_type_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_production_type", "id_parameter_production_type", "id_parameters_production_type", "code"),
    key_cols=["id_parameters_production_type"]
).withColumnRenamed("id_parameters_production_type", "id_parameter_production_type")

publish_dim(
    dim_trad_production_type, "dim_trad_production_type",
    ["id_parameter_production_type", "language"], current_catalog + "." + common_schema)

In [0]:
# parameters_variables_translations -> dim_trad_variable
dim_trad_variable = build_translation_dim(
    parameters_variables_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_variable", "id_parameter_variable", "parameter_variable", "code"),
    key_cols=["parameter_variable"]
).withColumnRenamed("parameter_variable", "id_parameter_variable")

publish_dim(
    dim_trad_variable, "dim_trad_variable", ["id_parameter_variable", "language"], current_catalog + "." + common_schema)

In [0]:
# parameters_production_line_variables_translations -> dim_trad_production_line_variable
dim_trad_production_line_variable = build_translation_dim(
    parameters_production_line_variables_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_production_line_variable", "id_parameter_production_line_variable", "parameter_production_line_variable", None),
    key_cols=["parameter_production_line_variable"]
).withColumnRenamed(
    "parameter_production_line_variable", "id_parameter_production_line_variable"
)

publish_dim(
    dim_trad_production_line_variable, "dim_trad_production_line_variable",
    ["id_parameter_production_line_variable", "language"], current_catalog + "." + common_schema)

In [0]:
# parameters_localizations_translations -> dim_trad_localization
dim_trad_localization = build_translation_dim(
    parameters_localizations_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_localization", "id_parameter_localization", "id_parameter_localization", "code"),
    key_cols=["id_parameter_localization"]
)

publish_dim(
    dim_trad_localization, "dim_trad_localization",
    ["id_parameter_localization", "language"], current_catalog + "." + common_schema)

In [0]:
# Contrôle : l'hypothèse "un seul libellé par (groupe, langue)" tient-elle ?
conflits_groupes = (
    parameters_localization_groups_translations
    .filter(F.col("deleted") == False)
    .groupBy("id_parameter_localization_group", "language")
    .agg(F.countDistinct("label").alias("nb_libelles"))
    .filter(F.col("nb_libelles") > 1)
)

if conflits_groupes.count() > 0:
    display(conflits_groupes)
    raise ValueError(
        "Des groupes de localisation ont plusieurs libellés selon la ligne de "
        "production. La clé de dim_trad_localization_group doit alors inclure "
        "production_line, et le modèle Power BI une colonne de substitution."
    )


dim_trad_localization_group = build_translation_dim(
    parameters_localization_groups_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_localization_group", "id_parameter_localization_group", "id_parameter_localization_group", "code"),
    key_cols=["id_parameter_localization_group"]
)

publish_dim(
    dim_trad_localization_group, "dim_trad_localization_group",
    ["id_parameter_localization_group", "language"], current_catalog + "." + common_schema)

In [0]:
# Le type de catégorie vient de la nomenclature (category_class), source faisant
# autorité, plutôt que d'un découpage du préfixe de la clé.
# La nomenclature conserve les noms source : sa clé est id_batch_note_category.
batch_note_categories = (
    spark.table(f"{source_catalog}.parameters_batch_note_categories")
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_batch_note_category").alias("batch_note_category"),
        F.col("category_class"),
        F.col("category_label"),
    )
)

batch_note_trad_base = (
    parameters_batch_note_categories_translations.alias("t")
    .join(
        F.broadcast(batch_note_categories).alias("n"),
        F.col("t.batch_note_category") == F.col("n.batch_note_category"),
        "inner"   # inner : une clé absente de la nomenclature n'a rien à traduire
    )
    .select("t.*", "n.category_class", "n.category_label")
)

if verbose_mode == 'debug':
    print("Traductions par type de catégorie :")
    display(
        batch_note_trad_base.filter(F.col("deleted") == False)
                            .groupBy("category_class").count().orderBy(F.desc("count"))
    )

In [0]:
BATCH_NOTE_TYPES = ["location", "event", "detail", "impact"]

for category_type in BATCH_NOTE_TYPES:
    df_type = batch_note_trad_base.filter(F.col("category_class") == category_type)

    dim_type = build_translation_dim(
        df_type,
        dim_trad_language,
        keys_df=cles_nomenclature(
            f"dim_batch_note_{category_type}",
            "id_batch_note_category", "batch_note_category",
            "category_label",
        ),
        key_cols=["batch_note_category"]
    )

    process_name = f"dim_trad_batch_note_{category_type}"
    globals()[process_name] = dim_type
    publish_dim(dim_type, process_name, ["batch_note_category", "language"], current_catalog + "." + common_schema)

In [0]:
MOIS = {
    "fr": ["Non renseigné", "janvier", "février", "mars", "avril", "mai", "juin",
           "juillet", "août", "septembre", "octobre", "novembre", "décembre"],
    "en": ["Not specified", "January", "February", "March", "April", "May", "June",
           "July", "August", "September", "October", "November", "December"],
    "ro": ["Nespecificat", "ianuarie", "februarie", "martie", "aprilie", "mai", "iunie",
           "iulie", "august", "septembrie", "octombrie", "noiembrie", "decembrie"],
    "cs": ["Neuvedeno", "leden", "únor", "březen", "duben", "květen", "červen",
           "červenec", "srpen", "září", "říjen", "listopad", "prosinec"],
}

# (culture_code, month_num, label) : l'indice 0 porte le libellé « non renseigné ».
lignes_mois = [
    (culture, num, libelle)
    for culture, libelles in MOIS.items()
    for num, libelle in enumerate(libelles)
]

mois_par_culture = spark.createDataFrame(
    lignes_mois, "culture_code string, month_num int, label string"
)

# La jointure sur dim_trad_language garantit qu'on n'écrit que les langues
# réellement actives, et qu'on récupère leur identifiant tel qu'il existe dans
# cet environnement.
dim_trad_month = (
    mois_par_culture.alias("m")
    .join(F.broadcast(dim_trad_language).alias("l"), "culture_code", "inner")
    .select(
        F.col("m.month_num"),
        F.col("l.language"),
        F.col("m.label"),
        F.lit("translated").alias("label_source"),
    )
)

attendu = 13 * dim_trad_language.count()
obtenu = dim_trad_month.count()
if obtenu != attendu:
    raise ValueError(
        f"dim_trad_month : {obtenu} lignes au lieu de {attendu}. "
        "Une langue de dim_trad_language n'a pas ses noms de mois dans MOIS."
    )

publish_dim(
    dim_trad_month, "dim_trad_month",
    ["month_num", "language"], current_catalog + "." + common_schema)


In [0]:
# processes_phases_translations -> dim_trad_processes_phases
dim_trad_processes_phases = build_translation_dim(
    processes_phases_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_processes_phases", "id_process_phase", "process_phase", "code"),
    key_cols=["process_phase"]
).withColumnRenamed("process_phase", "id_process_phase")

# Le code est rapatrie depuis la nomenclature : c'est lui qui porte la relation
# vers gold.phases[prd_workshop], l'identifiant n'etant pas present cote fait.
dim_trad_processes_phases = (
    dim_trad_processes_phases.alias("t")
    .join(
        spark.table(f"{current_catalog}.{common_schema}.dim_processes_phases")
             .filter(F.col("deleted") == False)
             .select(F.col("id_process_phase"), F.col("code")).alias("n"),
        "id_process_phase", "left",
    )
    .select("t.*", F.col("n.code"))
)

publish_dim(
    dim_trad_processes_phases, "dim_trad_processes_phases",
    ["id_process_phase", "language"], current_catalog + "." + common_schema)


In [0]:
# trs_translation -> dim_trad_trs_measure / _category / _measure_type
for entity_type in ["measure", "category", "measure_type"]:
    dim_trad_trs = build_translation_dim(
        trs_translations_forme_longue(trs_translation, dim_trad_language, entity_type),   # <= ICI
        dim_trad_language,
        keys_df=cles_nomenclature(f"dim_trs_{entity_type}", "key", "key", "key"),
        key_cols=["key"],
    )

    publish_dim(
        dim_trad_trs,
        f"dim_trad_trs_{entity_type}",
        ["key", "language"],
        current_catalog + "." + common_schema,
    )

In [0]:
# Le contrôle relit les tables Delta qui viennent d'être écrites, il ne rejoue pas
# les DataFrames. Deux raisons :
#   - performance : les DataFrames ne sont pas en cache, les réutiliser ferait
#     recalculer toute la chaîne depuis PostgreSQL (dédoublonnage, crossJoin,
#     jointures) et doublerait le temps du notebook ;
#   - fiabilité : on mesure ce qui est réellement en base, pas ce qui aurait dû
#     y être écrit.
couverture = reduce(
    lambda a, b: a.unionByName(b),
    [
        spark.table(target)
             .groupBy("language", "label_source")
             .count()
             .withColumn("table", F.lit(nom))
        for nom, target in tables_publiees
    ]
)

display(
    couverture.alias("c")
              .join(F.broadcast(dim_trad_language).alias("l"), "language", "left")
              .groupBy("table", "code")
              .pivot("label_source", ["translated", "fallback_en", "fallback_key"])
              .agg(F.sum("count"))
              .orderBy("table", "code")
)

In [0]:
jamais_traduites = reduce(
    lambda a, b: a.unionByName(b),
    [
        spark.table(target)
             .filter(F.col("label_source") == "fallback_key")
             .select(
                 F.lit(nom).alias("table"),
                 F.col("label").alias("code_affiche"),
             )
             .distinct()
        for nom, target in tables_publiees
    ]
)
jamais_traduites.cache()

print("Clés sans aucune traduction, par table :")
display(jamais_traduites.groupBy("table").count().orderBy(F.desc("count")))

print("Détail, à transmettre à l'équipe front :")
display(jamais_traduites.orderBy("table", "code_affiche"))